# Gaming Toxicity Detection

**Authors:** Beibarys Nyussupov, Ruby Ngo, Paola Calle, Jett Forward

In [1]:
# libraries 
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from pathlib import Path 
import re 

# reproducibility
seed = 7524
np.random.seed(seed)

# project root (notebooks/gaming/ - notebooks/ - project root)
PROJECT_ROOT = Path().resolve().parent.parent

In [2]:
# data directories
DATA_DIR_WOT  = PROJECT_ROOT / "data/raw_data/world_of_tanks/"
DATA_DIR_DOTA = PROJECT_ROOT / "data/raw_data/dota/"

## World of Tanks

**World of Tanks**
| Class | Terminology |
|---|---|
| 0 | Non-Toxic |
| 1 | Insults and Flaming |
| 2 | Other Offensive Texts |
| 3 | Hate and Harassment |
| 4 | Threats |
| 5 | Extremism |

#### Inspect each split

In [3]:
# import data 
train = pd.read_csv(DATA_DIR_WOT / "train.csv")
val = pd.read_csv(DATA_DIR_WOT / "val.csv")

# test dataset 
test_text = pd.read_csv(DATA_DIR_WOT /  "test_index_text.csv")
test_label = pd.read_csv(DATA_DIR_WOT / "test_index_label.csv")

# check the data 
for name, df in [("train", train), ("val", val), ("test_text", test_text), ("test_label", test_label)]:
    print(f"--- {name} ---")
    print(f"Shape: {df.shape}")
    print(f"Columns: {list(df.columns)}")
    print(f"First few rows of the dataset:\n{df.head(3)}")
    print()

--- train ---
Shape: (42959, 3)
Columns: ['index', 'message', 'label']
First few rows of the dataset:
   index                        message  label
0  30702                        no rush    0.0
1  18607  whatever ... watch the replay    0.0
2  32901                        useless    1.0

--- val ---
Shape: (5367, 3)
Columns: ['index', 'message', 'label']
First few rows of the dataset:
   index       message  label
0  31697  e50 is there    0.0
1  52311   this scouts    0.0
2   2775       i guess    0.0

--- test_text ---
Shape: (5375, 2)
Columns: ['index', 'message']
First few rows of the dataset:
   index         message
0  17775  aim you monkey
1  28228   Pz kill artys
2  19888             ggs

--- test_label ---
Shape: (5375, 2)
Columns: ['index', 'label']
First few rows of the dataset:
   index  label
0  17775    2.0
1  28228    0.0
2  19888    0.0



The test set is stored in two separate files - text and labels indexed by the same `index` column. Train and val are already complete. We need to join the test files on `index` before merging everything.

#### Reconstruct test split and merge all

In [4]:
# merge test text and labels
test = test_text.merge(test_label, on="index", how="inner")

# concatenate all data
wot = pd.concat([train, val, test], ignore_index=True)

print(f"Test join shape: {test.shape}\n")
print(f"Merged shape: {wot.shape}\n")
print(f"First few rows of the data:\n{wot.head(3)}\n")
print(f"Information about the data: {wot.info()}")

Test join shape: (5375, 3)

Merged shape: (53701, 3)

First few rows of the data:
   index                        message  label
0  30702                        no rush    0.0
1  18607  whatever ... watch the replay    0.0
2  32901                        useless    1.0

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 53701 entries, 0 to 53700
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   index    53701 non-null  int64  
 1   message  53701 non-null  object 
 2   label    53701 non-null  float64
dtypes: float64(1), int64(1), object(1)
memory usage: 1.2+ MB
Information about the data: None


The test join uses `inner` - any index present in only one file would be silently dropped. If test shape after join is smaller than either file's length, there is an alignment issue in the original data. 

#### Non-english language 

Since we are going to model classifier on english data, it is important for us to detect non-english cases as many as we can. 

In [5]:
# detecting Non-lating language first

# covers Cyrillic, CJK, Arabic, Hebrew, Thai, etc.
NON_LATIN_SCRIPT = re.compile(
    r"[\u0400-\u04FF"   # Cyrillic
    r"\u4E00-\u9FFF"    # CJK
    r"\u0600-\u06FF"    # Arabic
    r"\u0590-\u05FF"    # Hebrew
    r"\u3040-\u30FF"    # Japanese
    r"\u0E00-\u0E7F]"   # Thai
)

# function to detect non-lantin language 
def is_non_latin_script(text):
    # flag if >20% chars are non-Latin script
    chars = [c for c in str(text) if not c.isspace()]
    hits = sum(1 for c in chars if NON_LATIN_SCRIPT.match(c))
    return len(chars) > 0 and hits / len(chars) > 0.2

# non english wot 
print(f"Non-english documents:\n{wot[wot["message"].apply(is_non_latin_script)].reset_index(drop=True)}\n")
# drop such cases 
# length before drop 
before = len(wot)
wot_en = wot[~wot["message"].apply(is_non_latin_script)].reset_index(drop=True)
print(f"Dropped {before - len(wot_en)} ({(before-len(wot_en))/before:.1%})")
print(f"Rows after the drop: {wot_en.shape[0]}")

Non-english documents:
      index                message  label
0     13087          ще злийте )))    1.0
1     31917       перший бій, сток    0.0
2     40395                ФОКУСЯТ    0.0
3     54412  елка спасибо тебе ...    0.0
4      5457                 уроди!    0.0
...     ...                    ...    ...
3732   5453                 я кину    0.0
3733   5456                светите    0.0
3734  13081       в чём притензия?    0.0
3735   7331      вот и подкруточки    0.0
3736  50128  тут танки стоят уебак    0.0

[3737 rows x 3 columns]

Dropped 3737 (7.0%)
Rows after the drop: 49964


In [7]:
from lingua import Language, LanguageDetectorBuilder

# detector to find latin but non-english documents 
detector = (LanguageDetectorBuilder 
    # detect all other languages using latin
    .from_all_languages_with_latin_script()
    # if top-2 languages are within 25% confidence of each other - do not drop 
    .with_minimum_relative_distance(0.25)
    .build())

# function to detect such documents 
def is_latin_non_english(text):
    if len(str(text).strip()) < 30:
        return False  # too short - keep
    lang = detector.detect_language_of(str(text))
    return lang is not None and lang != Language.ENGLISH

# such documents 
mask = wot_en["message"].apply(is_latin_non_english)
with pd.option_context("display.max_rows", 800):
    print(f"Non-english documents:\n{wot_en[mask]}\n")
    print(f"How many documents with non-english text:{wot_en[mask].shape[0]}")

Non-english documents:
       index                                            message  label
364    14038                    Menj már nemm kapsz spotot.....    0.0
494    29761          no ayudaste a nadie shptk ahora no llores    1.0
573     7353                  3 dingi od pierdolonych papier??w    2.0
683    35518                Etwas Hilfe bei D4 wäre schon nett!    0.0
779     6397        Hilfe! Ich werde gleich auf A3 verpr??gelt!    0.0
908     6395        Hilfe! Ich werde gleich auf C1 verpr??gelt!    0.0
958     1934  jeszcze t54 zespotowany byl wczesniej wiec wid...    0.0
1110    7645                     how many dmg arty? 2k? XDDDDDD    0.0
1111   51836                   LOSSSSSSSSSSSSSSSSSSSSSSSSSSSSSS    0.0
1185    1984                   5x ep... mehr muss man net sagen    0.0
1374    6449  Hali! A MAATT kl??n akt??v j??t??kosokat keres...    0.0
1426   30904                  ooo polaczek, no jakby inaczej XD    0.0
1528    5943  tu ecris ce que tu veux, dans la mesure 

Only 294 rows. Let's drop that as well. 


In [8]:
# dataset update
wot_en = wot_en[~mask].reset_index(drop = True)
# length after the drop 
print(f"Length after the drop:{wot_en.shape[0]}")

Length after the drop:49670


#### Mislabeled data 

In [9]:
# few top rows of the data
wot_en.head(5)

,index,message,label
0,30702,no rush,0.0
1,18607,whatever ... watch the replay,0.0
2,32901,useless,1.0
3,25964,3 gunmark,0.0
4,28643,lol,0.0


We might have mislabeled toxicity data that was classified as non-toxic or opposite. We need to detect and fix that as well. 

In [12]:
from cleanlab.filter import find_label_issues
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import cross_val_predict

# splits 
X = wot_en["message"].values
y = wot_en["label"].astype(int).values

# simple pipeline 
pipe = Pipeline([
    ("tfidf", TfidfVectorizer(ngram_range=(1,2), min_df=1, max_df=0.95,
                               sublinear_tf=True, norm="l2")),
    ("clf", LogisticRegression(max_iter=1000, random_state=7524)),
])

# out-of-fold predicted probabilities (needs predict_proba - LR not LinearSVC)
oof_probs = cross_val_predict(pipe, X, y, cv=5, method="predict_proba", n_jobs=-1)

# find label issues
issue_idx = find_label_issues(
    labels=y,
    pred_probs=oof_probs,
    return_indices_ranked_by="self_confidence",  # worst first
)

print(f"Suspected mislabeled: {len(issue_idx)} ({len(issue_idx)/len(y):.1%})")

# inspect top offenders
wot_en.iloc[issue_idx[:50]][["message", "label"]].assign(
    predicted=oof_probs[issue_idx[:50]].argmax(axis=1)
)

Suspected mislabeled: 2511 (5.1%)


,message,label,predicted
44060,idiot campersw,0.0,1
42775,idiot,0.0,1
40041,idiot,0.0,1
42189,idiot,0.0,1
28059,idiot,0.0,1
25302,idiot,0.0,1
10128,cromvel idiot,0.0,1
13752,idiot,0.0,1
12624,idiot,0.0,1
37867,idiot,0.0,1


This is noisy. We need to detect those that were not classified toxic while toxic


In [13]:
predicted = oof_probs.argmax(axis=1)

# only flag: labeled non-toxic, model confidently says toxic
non_toxic_mislabeled = (
    (y == 0) &                          # labeled non-toxic
    (predicted != 0) &                  # model says toxic
    (oof_probs.max(axis=1) > 0.60)      # let's explore for any confidence 
)

# filter and sort to flagged 
flagged = wot_en[non_toxic_mislabeled][["message", "label"]].copy()
flagged["predicted"] = predicted[non_toxic_mislabeled]
flagged["confidence"] = oof_probs.max(axis=1)[non_toxic_mislabeled]
flagged = flagged.sort_values("confidence", ascending=False)

with pd.option_context("display.max_rows", 595):
    print(f"Suspected 0 - toxic mislabels: {len(flagged)}")
    print(flagged.head(594))

Suspected 0 - toxic mislabels: 459
                                                 message  label  predicted  \
44060                                     idiot campersw    0.0          1   
42775                                              idiot    0.0          1   
42189                                              idiot    0.0          1   
40041                                              idiot    0.0          1   
13752                                              idiot    0.0          1   
10128                                      cromvel idiot    0.0          1   
12624                                              idiot    0.0          1   
28059                                              idiot    0.0          1   
25302                                              idiot    0.0          1   
37867                                              idiot    0.0          1   
14438                                                bot    0.0          1   
11261                        

Most of them are mislabeled. Let's drop those 


In [14]:
wot_en = wot_en[~non_toxic_mislabeled].reset_index(drop = True)
print(f"Length after update:{wot_en.shape[0]}")

Length after update:49211


#### Final verification

In [15]:
# basic data checks
print("=== Shape ===")
print(wot_en.shape)

print("\n=== Nulls ===")
print(wot_en.isnull().sum())

print("\n=== Duplicate messages ===")
print(wot_en.duplicated(subset="message").sum())

print("\n=== Overall label distribution ===")
counts = wot_en["label"].value_counts().sort_index()
pct = wot_en["label"].value_counts(normalize=True).sort_index().mul(100).round(1)
print(pd.DataFrame({"count": counts, "pct%": pct}))

=== Shape ===
(49211, 3)

=== Nulls ===
index      0
message    0
label      0
dtype: int64

=== Duplicate messages ===
18040

=== Overall label distribution ===
       count  pct%
label             
0.0    39950  81.2
1.0     6739  13.7
2.0     2177   4.4
3.0      252   0.5
4.0       66   0.1
5.0       27   0.1


We do not have any null values, however we got 18k duplicate messages.

We will explore duplicate rows and other issues in EDA. For now, let's save merged dataset to a parquet. 



In [16]:
# save to parquet for easier loading in future steps
wot_en.to_parquet(PROJECT_ROOT / "data/processed_data/wot/wot.parquet", index=False)

## Dota

**Dota**
| Class | Label |
|---|---|
| 0 | Other (non-toxic) |
| 1 | Ego |
| 2 | Aggression |
| 3 | Impolite |

#### Inspect each split

In [17]:
# import data
train = pd.read_csv(DATA_DIR_DOTA / "CONDA_train.csv")
val = pd.read_csv(DATA_DIR_DOTA / "CONDA_valid.csv")

# inspect each split
for name, df in [("train", train), ("val", val)]:
    print(f"--- {name} ---")
    print(f"Shape: {df.shape}\n")
    print(f"Columns: {list(df.columns)}\n")
    print(f"First few rows of the data:\n{df.head(3)}\n")

--- train ---
Shape: (26921, 10)

Columns: ['Id', 'matchId', 'conversationId', 'utterance', 'chatTime', 'playerSlot', 'playerId', 'intentClass', 'slotClasses', 'slotTokens']

First few rows of the data:
      Id  matchId  conversationId utterance  chatTime  playerSlot  \
0  11263      697            3193      wow!        76           0   
1  13741      843            3809       WTF      1563           5   
2  22125     1412            6199   wpe wpe      2853           1   

                  playerId intentClass slotClasses          slotTokens  
0  ANTS IN MY EYES JOHNSON           O          O            wow (O),   
1                      M.k           O          T            WTF (T),   
2              Acqua Ragia           O        O O   wpe (O), wpe (O),   

--- val ---
Shape: (8974, 10)

Columns: ['Id', 'matchId', 'conversationId', 'utterance', 'chatTime', 'playerSlot', 'playerId', 'intentClass', 'slotClasses', 'slotTokens']

First few rows of the data:
      Id  matchId  conversa

#### Clean, merge and standardise

In [18]:
# map intentClass to numeric label consistent with WoT schema
label_map = {'O': 0, 'E': 1, 'A': 2, 'I': 3}

# add split column and map labels
train["split"] = "train"
val["split"]   = "val"

# merge train + val (no test split in CONDA)
dota = pd.concat([train, val], ignore_index=True)

# keep only relevant columns and rename to match WoT schema
dota = dota[["Id", "utterance", "intentClass", "split"]].rename(columns={
    "Id": "index",
    "utterance":   "message",
    "intentClass": "label",
})

# map string labels to numeric
dota["label"] = dota["label"].map(label_map)

# drop nulls in message (7 train + 1 val)
dota = dota.dropna(subset=["message"]).reset_index(drop=True)

# shape 
print(f"Merged shape: {dota.shape}\n")
print(f"First few rows of the data:\n{dota.head(3)}\n")
print("\nInformation about the dataset:\n")
dota.info()

Merged shape: (35887, 4)

First few rows of the data:
   index  message  label  split
0  11263     wow!      0  train
1  13741      WTF      0  train
2  22125  wpe wpe      0  train


Information about the dataset:

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 35887 entries, 0 to 35886
Data columns (total 4 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   index    35887 non-null  int64 
 1   message  35887 non-null  object
 2   label    35887 non-null  int64 
 3   split    35887 non-null  object
dtypes: int64(2), object(2)
memory usage: 1.1+ MB


Only `utterance`, `intentClass`, `Id`, and `split` are kept = the remaining columns (matchId, conversationId, chatTime, playerSlot, playerId, slotClasses, slotTokens) carry no linguistic content relevant to toxicity detection. Labels are mapped to integers matching the WoT schema where 0 = non-toxic. Null utterances are dropped - they carry no text signal.

#### Non-english language 

Since we are going to model classifier on english data, it is important for us to detect non-english cases as many as we can. 

In [19]:
# detecting Non-latin language first - same regex as WoT
dota_en_mask = dota["message"].apply(is_non_latin_script)
print(f"Non-english documents: {dota[dota_en_mask].reset_index(drop=True)}")

# drop such cases
before = len(dota)
dota_en = dota[~dota_en_mask].reset_index(drop=True)
print(f"Dropped {before - len(dota_en)} ({(before - len(dota_en))/before:.1%})")
print(f"Rows after the drop: {dota_en.shape[0]}")

Non-english documents:    index          message  label  split
0  36141  noob [SEPA] กาก      1  train
1  19064            ใจยเน      0  train
2   8391                ツ      0  train
3  27048                ื      0  train
4  37396               ำผ      0  train
5  41950              กาก      0    val
6  19066            คนไทย      0    val
7  32824               ทำ      0    val
8  33829              戒了吧      0    val
Dropped 9 (0.0%)
Rows after the drop: 35878


In [21]:
# detect latin but non-english documents - same detector as WoT
mask_d = dota_en["message"].apply(is_latin_non_english)
with pd.option_context("display.max_rows", 800):
    print(f"Non-english documents:{dota_en[mask_d]}")
    print(f"How many documents with non-english text: {dota_en[mask_d].shape[0]}")

Non-english documents:       index                                            message  label  split
1189   38131  reporta [SEPA] pro ibama [SEPA] fdp sacrifica ...      2  train
1196   40006  its in the bag  [SEPA] los q no tienen plata [...      0  train
1594   27145  vcs sao lixos [SEPA] n fode [SEPA] acha q fez ...      0  train
2394   38352  pukingina ka [SEPA] kangkong lancer [SEPA] goo...      0  train
4138     466  REPORT MI TEAM [SEPA] REPORT TODO MI TEAM BURR...      2  train
5765   12964                   LOOOOOOOOOL [SEPA] RECOMIENDENME      0  train
6387   37068                   oyoyoyoyoyoyoy wombo combo......      0  train
6472   25399  pro farmer [SEPA] orge farmt mehr als jeder bauer      0  train
6866   42190  very [SEPA] report a los 4 les puede tocar est...      2  train
6942   41505                 ez [SEPA] ez [SEPA] ez [SEPA] ezwp      3  train
8741   34169  POTANG INA KAYO [SEPA] BIGYAN NIYO NAMAN AKO N...      0  train
9141   33088  OSfrog LE BALANCED FEMALE RO

Only 56 rows. Let's drop that as well. 

In [22]:
# dataset update
dota_en = dota_en[~mask_d].reset_index(drop=True)
# length after the drop
print(f"Length after the drop: {dota_en.shape[0]}")

Length after the drop: 35822


#### Mislabeled data 

In [23]:
# few top rows of the data
dota_en.head(5)

,index,message,label,split
0,11263,wow!,0,train
1,13741,WTF,0,train
2,22125,wpe wpe,0,train
3,6453,hahaha,0,train
4,9644,wtf,0,train


We might have mislabeled toxicity data that was classified as non-toxic or opposite. We need to detect and fix that as well. 

In [24]:
# splits
X_d = dota_en["message"].values
y_d = dota_en["label"].astype(int).values

# simple pipeline - same config as WoT
pipe_d = Pipeline([
    ("tfidf", TfidfVectorizer(ngram_range=(1,2), min_df=1, max_df=0.95,
                               sublinear_tf=True, norm="l2")),
    ("clf", LogisticRegression(max_iter=1000, random_state=7524)),
])

# out-of-fold predicted probabilities
oof_probs_d = cross_val_predict(pipe_d, X_d, y_d, cv=5, method="predict_proba", n_jobs=-1)

# find label issues
issue_idx_d = find_label_issues(
    labels=y_d,
    pred_probs=oof_probs_d,
    return_indices_ranked_by="self_confidence",
)

print(f"Suspected mislabeled: {len(issue_idx_d)} ({len(issue_idx_d)/len(y_d):.1%})")

# inspect top offenders
dota_en.iloc[issue_idx_d[:50]][["message", "label"]].assign(
    predicted=oof_probs_d[issue_idx_d[:50]].argmax(axis=1)
)

Suspected mislabeled: 1527 (4.3%)


,message,label,predicted
7706,EZ^,0,3
7162,EZ),0,3
5386,ez&,0,3
4696,ez,0,3
25633,NOOB!,0,1
5070,noob,0,1
30173,NOOB,0,1
5468,shit,0,1
17515,SHUT UP FUCKING NOOB !,0,1
14254,EZ PERRUUUUUUU,1,3


This is again noisy. We need to detect those that were not classified toxic while toxic

In [25]:
predicted_d = oof_probs_d.argmax(axis=1)

# only flag: labeled non-toxic, model confidently says toxic
non_toxic_mislabeled_d = (
    (y_d == 0) &
    (predicted_d != 0) &
    (oof_probs_d.max(axis=1) > 0.60)
)

# filter and sort to flagged
flagged_d = dota_en[non_toxic_mislabeled_d][["message", "label"]].copy()
flagged_d["predicted"] = predicted_d[non_toxic_mislabeled_d]
flagged_d["confidence"] = oof_probs_d.max(axis=1)[non_toxic_mislabeled_d]
flagged_d = flagged_d.sort_values("confidence", ascending=False)

with pd.option_context("display.max_rows", 252):
    print(f"Suspected 0 - toxic mislabels: {len(flagged_d)}")
    print(flagged_d)

Suspected 0 - toxic mislabels: 252
                                                 message  label  predicted  \
25633                                              NOOB!      0          1   
30173                                               NOOB      0          1   
5070                                                noob      0          1   
5468                                                shit      0          1   
7706                                                 EZ^      0          3   
7162                                                 EZ)      0          3   
17515                             SHUT UP FUCKING NOOB !      0          1   
5386                                                 ez&      0          3   
4696                                                  ez      0          3   
10033                                     okay fuck shit      0          1   
34458                           Fuck [SEPA] fucking zeus      0          1   
22689                        

Pattern clear: false positives cluster in predicted=2 (Aggression) - "commend", "wait", "pause", "afk" - model confuses coordination messages with aggression. Genuine mislabels mostly in predicted=1 (Ego/Insults): "noob", "fuck", "stfu", "ez".

Fix: only flag label=0 - predicted=1 (Insults). Skip predicted=2 (Aggression) - too noisy for Dota:

In [26]:
non_toxic_mislabeled_d = (
    (y_d == 0) &
    (predicted_d == 1) &        # only Ego/Insults, not Aggression
    (oof_probs_d.max(axis=1) > 0.60)
)

# filter and sort to flagged
flagged_d = dota_en[non_toxic_mislabeled_d][["message", "label"]].copy()
flagged_d["predicted"] = predicted_d[non_toxic_mislabeled_d]
flagged_d["confidence"] = oof_probs_d.max(axis=1)[non_toxic_mislabeled_d]
flagged_d = flagged_d.sort_values("confidence", ascending=False)

with pd.option_context("display.max_rows", 252):
    print(f"Suspected 0 - toxic mislabels: {len(flagged_d)}")
    print(flagged_d)

Suspected 0 - toxic mislabels: 131
                                                 message  label  predicted  \
25633                                              NOOB!      0          1   
30173                                               NOOB      0          1   
5070                                                noob      0          1   
5468                                                shit      0          1   
17515                             SHUT UP FUCKING NOOB !      0          1   
10033                                     okay fuck shit      0          1   
34458                           Fuck [SEPA] fucking zeus      0          1   
28437                                               fuck      0          1   
27120                                               fuck      0          1   
22689                                               fuck      0          1   
26573                                               fuck      0          1   
10860                        

Let's drop those 

In [27]:
# drop mislabelling 
dota_en = dota_en[~non_toxic_mislabeled_d].reset_index(drop=True)
print(f"Length after update: {dota_en.shape[0]}")

Length after update: 35691


#### Final verification

In [28]:
print("=== Shape ===")
print(dota_en.shape)

print("=== Nulls ===")
print(dota_en.isnull().sum())

print("=== Duplicate messages ===")
print(dota_en.duplicated(subset="message").sum())

print("=== Label distribution ===")
counts = dota_en["label"].value_counts().sort_index()
pct = dota_en["label"].value_counts(normalize=True).sort_index().mul(100).round(1)
label_names = {0: "Other (O)", 1: "Ego (E)", 2: "Aggression (A)", 3: "Impolite (I)"}
print(pd.DataFrame({"count": counts, "pct%": pct}).rename(index=label_names))

=== Shape ===
(35691, 4)
=== Nulls ===
index      0
message    0
label      0
split      0
dtype: int64
=== Duplicate messages ===
11051
=== Label distribution ===
                count  pct%
label                      
Other (O)       26424  74.0
Ego (E)          4703  13.2
Aggression (A)   2292   6.4
Impolite (I)     2272   6.4


No null values after dropping 8 empty utterances. Duplicates exist but will be explored in EDA - short gaming phrases like "gg" appear frequently across matches and are expected repeats, not data errors. Label distribution mirrors WoT: heavy skew toward non-toxic (O), with Ego, Aggression, and Impolite as minority classes. Save to parquet.

In [29]:
# save cleaned dota to parquet for easier loading in future steps
dota_en.to_parquet(PROJECT_ROOT / "data/processed_data/dota/dota.parquet", index=False)